# Learned masked-image reconstruction

**MedLIME Learning Track — Notebook 06/12** · Curriculum weeks 21–24 · Suggested time: 60–100 minutes

**Before starting:** Notebook 05; beginner PyTorch experience.

**Learning goal:** Train a lightweight self-supervised masked-image reconstructor and audit hallucination.

> **Important:** All images below are synthetic visual teaching phantoms, **not real medical images**. The educational implementations are **not** the original MedLIME implementation, are not evaluated for clinical use, and do not establish diagnostic or explanation reliability.


In [ ]:
from pathlib import Path
import sys
here = Path.cwd()
for candidate in (here, here.parent, here.parent.parent):
    if (candidate / 'medlime_tutorials.py').exists():
        sys.path.insert(0, str(candidate.resolve())); break
else:
    raise FileNotFoundError('Open Jupyter from the MedLIME folder or its notebooks/ subfolder')
import numpy as np
import matplotlib.pyplot as plt
from medlime_tutorials import *
np.set_printoptions(precision=3, suppress=True)


## From masking to reconstruction
Generative masking in MedLIME is motivated by avoiding unnatural holes. A masked autoencoder learns to predict missing image content using observed context. Our **tiny convolutional inpainter is NOT a masked-autoencoder transformer**: it teaches the core input/target/loss workflow on CPU before students study MAE papers and implementations.

Training target: original image \(x\); observed input: \(m\odot x\). The target includes no radiologist diagnosis.

In [ ]:
model,losses=train_inpainter(n=96,epochs=8,seed=10)
plt.plot(losses,marker='o');plt.xlabel('Epoch');plt.ylabel('Last minibatch weighted MSE');plt.title('Self-supervised reconstruction training');plt.show()
print('Model class:',type(model).__name__)

## Does the model preserve known information?
Observed pixels are copied from the original. Only hidden pixels come from predictions. Compare reconstruction error to simple baselines, not just visual plausibility.

In [ ]:
x,m=synthetic_image(seed=40)
keep=np.ones(16);keep[[5,6,9,10]]=0
reconstructed=inpaint(x,keep,model)
black=patch_mask(x,keep,fill='zero')
blur=patch_mask(x,keep,fill='blur')
show_images([x,black,blur,reconstructed],['original','black fill','blur fill','learned reconstruction'])
plt.show()
hidden=(keep[patch_ids()]==0)
for name,z in [('zero',black),('blur',blur),('learned',reconstructed)]:
 print(name,'masked-region MSE:',round(float(np.mean((x[hidden]-z[hidden])**2)),5))
assert np.allclose(x[~hidden],reconstructed[~hidden])

## A critical limitation
A reconstructor may erase a lesion that was present in a masked patch, or hallucinate a bright spot that was absent. Visual naturalness is *not* proof that a generated image preserves pathology. This is central to MedLIME's scientific interpretation.

In [ ]:
visible_lesion=np.count_nonzero(m & ~hidden)
hidden_lesion=np.count_nonzero(m & hidden)
print('Lesion pixels still observed:',visible_lesion,'lesion pixels hidden:',hidden_lesion)
print('Average intensity in hidden lesion before/after:',
      round(float(x[m & hidden].mean()),3) if hidden_lesion else 'not hidden',
      round(float(reconstructed[m & hidden].mean()),3) if hidden_lesion else 'not hidden')

## What you should now understand
- Self-supervised learning can use images as their own reconstruction targets.
- Missing-pixel MSE tests numerical accuracy; realism and pathology preservation need other tests.
- A small CNN inpainter teaches mechanics but does not replicate the MedLIME MAE.

## Try it yourself (do not just read the code)
1. Use a different mask to hide the complete synthetic lesion and record whether it returns.
2. Train for 2 versus 12 epochs; compare masked-region error on a held-out image.

## Exit ticket
**Question:** Why can very good reconstruction MSE still correspond to a poor explanation of a rare lesion?

Record your answers in a new Markdown cell before moving to the next notebook. Many questions have more than one reasonable answer; explain your evidence.